In [ ]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

## Introduction, Task Brief and Data Dictionary

This project places us in the role of a data analyst within a real estate company. We have been assigned to generate an analysis on crime in the UK. Our stakeholder is Nadine Green, Head of Sales. Nadine is keen to know which locations are most / least desirable and any useful information for her to make
informed decisions around sales.

Nadine Green has requested this analysis to have a general idea of the crime across the regions. The outcome of this should be the selection of two police forces for further study (to do in a later stage).

### Data Dictionary
| Column Name               | Data Type            | Description                                                                                                                       | Example                  |
| ------------------------- | -------------------- | --------------------------------------------------------------------------------------------------------------------------------- | ------------------------ |
| **Crime ID**              | String               | Unique identifier assigned to each recorded crime incident.                                                                       | `a1b2c3d4`               |
| **Month**                 | Date (YYYY-MM)       | The month when the crime was recorded by the police force.                                                                        | `2024-01`                |
| **Reported by**           | Categorical (String) | The police force that recorded the crime incident.                                                                                | `West Midlands Police`   |
| **Falls within**          | Categorical (String) | The police jurisdiction responsible for the geographic area where the crime occurred.                                             | `Thames Valley Police`   |
| **Longitude**             | Float                | Longitude coordinate of the crime location in decimal degrees (WGS84).                                                            | `-1.2577`                |
| **Latitude**              | Float                | Latitude coordinate of the crime location in decimal degrees (WGS84).                                                             | `52.4068`                |
| **Location**              | String               | Approximate location of the crime, typically anonymised to protect privacy.                                                       | `On or near High Street` |
| **LSOA code**             | String               | Code identifying the **Lower Layer Super Output Area (LSOA)** where the crime occurred. Used for small-area statistical analysis. | `E01012345`              |
| **LSOA name**             | String               | Human-readable name of the LSOA corresponding to the code.                                                                        | `Oxford 012B`            |
| **Crime type**            | Categorical (String) | Classification of the crime (e.g., burglary, vehicle crime, anti-social behaviour).                                               | `Burglary`               |
| **Last outcome category** | Categorical (String) | Final recorded outcome of the investigation for the crime incident.                                                               | `Under investigation`    |

Notes:
LSOAs are small geographic areas used for statistical analysis in England and Wales. Each LSOA typically contains ~1,500 residents.

## Data Importing

The data importing section has been designed according to the folder structure when downloading data from the [UK police crime data](https://data.police.uk/data/) website. Since the data is split into months, the script first converts each .csv into pandas dataframes, then concatenates them based on police region using the "regions" list. The end result is four dataframes which can now be proccessed in Python.

In [4]:
base_path = "a5c44fe1462950800f6e6bf002dd3d82943a303b"

regions = [
    "thames-valley",
    "warwickshire",
    "west-midlands",
    "west-yorkshire"
]

# dictionary to store monthly dataframes for each region
region_data = {region: [] for region in regions}

for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)

    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            print(file)
            
            for region in regions:
                if region in file and file.endswith(".csv"):
                    file_path = os.path.join(folder_path, file)
                    
                    df = pd.read_csv(file_path)
                    region_data[region].append(df)

# merge into final dataframes
thames_valley_df = pd.concat(region_data["thames-valley"], ignore_index=True)
warwickshire_df = pd.concat(region_data["warwickshire"], ignore_index=True)
west_midlands_df = pd.concat(region_data["west-midlands"], ignore_index=True)
west_yorkshire_df = pd.concat(region_data["west-yorkshire"], ignore_index=True)

2024-09-west-midlands-street.csv
2024-09-warwickshire-street.csv
2024-09-thames-valley-street.csv
2024-09-west-yorkshire-street.csv
2024-07-thames-valley-street.csv
2024-07-west-yorkshire-street.csv
2024-07-warwickshire-street.csv
2024-07-west-midlands-street.csv
2024-06-thames-valley-street.csv
2024-06-warwickshire-street.csv
2024-06-west-yorkshire-street.csv
2024-06-west-midlands-street.csv
2024-01-west-midlands-street.csv
2024-01-warwickshire-street.csv
2024-01-thames-valley-street.csv
2024-01-west-yorkshire-street.csv
2024-08-west-yorkshire-street.csv
2024-08-west-midlands-street.csv
2024-08-warwickshire-street.csv
2024-08-thames-valley-street.csv
2025-06-thames-valley-street.csv
2025-06-warwickshire-street.csv
2025-06-west-yorkshire-street.csv
2025-06-west-midlands-street.csv
2025-01-warwickshire-street.csv
2025-01-west-midlands-street.csv
2025-01-west-yorkshire-street.csv
2025-01-thames-valley-street.csv
2025-08-west-midlands-street.csv
2025-08-thames-valley-street.csv
2025-08-we

## Data Integrity Checking
With the historical data of all four police forces successfully aggregated, a standardised data integrity checking procedure will be used to discover and investigate any anomalies within the dataset.

The stages are as follows:
| Stage | Method | Description / Purpose |
|------|------|------|
| 1 | `df.head()` | Gives preview of first 5 entries in df to ensure data has been correctly imported |
| 2 | `df.tail()` | Gives preview of last 5 entries in df to ensure data has been correctly imported |
| 3 | `df.info()` | Provides key information of the dataframe such as shape, column names, count of non-null values, and memory usage |
| 4 | `df.isna().sum()` | Counts the number of NA values for each column which will be used for the data cleaning stage |
| 5 | `df.describe()` | Calculates basic statistical measures for numerical columns to check for outliers |
| 6 | `df.nunique()` | Counts the unique values that exist in each column, providing insight into what types of aggregations could be possible in the EDA stage |
| 7 | `df['Crime type'].unique()` | Identifying the specific unique values in the `Crime type` column to understand what values this column can take. Also identifies areas to explore for the EDA stage |
| 8 | `df['Last outcome category'].unique()` | Identifying the specific unique values in the `Last outcome category` column to understand what values this column can take. Also identifies areas to explore for the EDA stage |

Note that these steps have been repeated when one could concatenate all four dataframes to check for data integrity. The reason for this is that this project aims to treat each police region independently throughout as the final outcome of the project is the selection of two police forces for further study. Additionaly, certain data issues may only be unique to a certain police region and seperation makes it much clearer when identifying these issues.

### Thames Valley

In [5]:
# thames_valley_df = thames_valley_df.sort_values("Month")
thames_valley_df.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,600dc88bebb3499405cf71237c6a89fa4f0e94f51d669d...,2024-09,Thames Valley Police,Thames Valley Police,-0.704117,51.448857,On or near Winkfield Lane,E01016252,Bracknell Forest 001C,Burglary,Investigation complete; no suspect identified,NaN
1,96339937ce33810cdd4421d7960675e206d9a57e8cddd7...,2024-09,Thames Valley Police,Thames Valley Police,-0.671521,51.446882,On or near Montague Park,E01016252,Bracknell Forest 001C,Other theft,Unable to prosecute suspect,NaN
2,48064c2c5c76557638c7d6e9351d59dd3c6a9105a042d5...,2024-09,Thames Valley Police,Thames Valley Police,-0.667887,51.450114,On or near Park Lane,E01016252,Bracknell Forest 001C,Other theft,Status update unavailable,NaN
3,b88c4f1501e108884efed2d65a1affd624539250adb7c9...,2024-09,Thames Valley Police,Thames Valley Police,-0.672384,51.435490,On or near Lovel Lane,E01016252,Bracknell Forest 001C,Other theft,Investigation complete; no suspect identified,NaN
4,a8481f9aa2db34e456f19bdbd0620a9e646cb7f98d55a9...,2024-09,Thames Valley Police,Thames Valley Police,-0.677236,51.436336,On or near Lovel Road,E01016252,Bracknell Forest 001C,Violence and sexual offences,Unable to prosecute suspect,NaN


In [6]:
thames_valley_df.tail()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
408438,24df801e9c0a9c2867730e20d943967352bd0ec9b1cfba...,2024-10,Thames Valley Police,Thames Valley Police,NaN,NaN,No Location,NaN,NaN,Other crime,Unable to prosecute suspect,NaN
408439,88000749532504ec8e573ea2f8677fd4a171a65ea16805...,2024-10,Thames Valley Police,Thames Valley Police,NaN,NaN,No Location,NaN,NaN,Other crime,Unable to prosecute suspect,NaN
408440,a9506edf48768c0dbc14139c7dcab49e45079ddd0704a5...,2024-10,Thames Valley Police,Thames Valley Police,NaN,NaN,No Location,NaN,NaN,Other crime,Unable to prosecute suspect,NaN
408441,07f123fe6cf4068577006cfb20c9eb5e262f584ccc1702...,2024-10,Thames Valley Police,Thames Valley Police,NaN,NaN,No Location,NaN,NaN,Other crime,Court result unavailable,NaN
408442,ae9c0055564d72783428fb0c1b2bb3d21ed9438e93d5cb...,2024-10,Thames Valley Police,Thames Valley Police,NaN,NaN,No Location,NaN,NaN,Other crime,Status update unavailable,NaN


In [7]:
thames_valley_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 408443 entries, 0 to 408442
Data columns (total 12 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Crime ID               366697 non-null  str    
 1   Month                  408443 non-null  str    
 2   Reported by            408443 non-null  str    
 3   Falls within           408443 non-null  str    
 4   Longitude              399558 non-null  float64
 5   Latitude               399558 non-null  float64
 6   Location               408443 non-null  str    
 7   LSOA code              399558 non-null  str    
 8   LSOA name              399558 non-null  str    
 9   Crime type             408443 non-null  str    
 10  Last outcome category  366697 non-null  str    
 11  Context                0 non-null       float64
dtypes: float64(3), str(9)
memory usage: 37.4 MB


In [8]:
thames_valley_df.isna().sum()

Crime ID                  41746
Month                         0
Reported by                   0
Falls within                  0
Longitude                  8885
Latitude                   8885
Location                      0
LSOA code                  8885
LSOA name                  8885
Crime type                    0
Last outcome category     41746
Context                  408443
dtype: int64

In [9]:
thames_valley_df.describe()

,Longitude,Latitude,Context
count,399558.000000,399558.000000,0.0
mean,-0.931527,51.678748,NaN
std,0.271489,0.224645,NaN
min,-1.685051,51.336336,NaN
25%,-1.201635,51.476538,NaN
50%,-0.828534,51.630797,NaN
75%,-0.731907,51.824564,NaN
max,-0.481458,52.190402,NaN


In [10]:
thames_valley_df_summary = thames_valley_df.nunique().reset_index(name='n_unique')
thames_valley_df_summary.columns = ['column', 'n_unique']

print(thames_valley_df_summary)

                   column  n_unique
0                Crime ID    366559
1                   Month        25
2             Reported by         1
3            Falls within         1
4               Longitude     29693
5                Latitude     29488
6                Location     20383
7               LSOA code      1490
8               LSOA name      1490
9              Crime type        14
10  Last outcome category        14
11                Context         0


In [11]:
for x in thames_valley_df['Crime type'].unique():
     print(x)

Burglary
Other theft
Violence and sexual offences
Possession of weapons
Other crime
Public order
Criminal damage and arson
Anti-social behaviour
Bicycle theft
Drugs
Shoplifting
Robbery
Vehicle crime
Theft from the person


In [12]:
for x in thames_valley_df['Last outcome category'].unique():
    print(x)

Investigation complete; no suspect identified
Unable to prosecute suspect
Status update unavailable
Action to be taken by another organisation
Awaiting court outcome
Court result unavailable
nan
Further investigation is not in the public interest
Local resolution
Offender given a caution
Suspect charged as part of another case
Further action is not in the public interest
Offender given penalty notice
Formal action is not in the public interest
Under investigation


### Warwickshire 

In [13]:
warwickshire_df.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,79d40667dd3ade5f202d0cfa5295777a869d9a30362fe2...,2024-09,Warwickshire Police,Warwickshire Police,-1.774147,52.494482,On or near Coneyford Road,E01009306,Birmingham 046B,Public order,Status update unavailable,NaN
1,236de2cd796df699d95982495b710ad2f35c4ac26db31f...,2024-09,Warwickshire Police,Warwickshire Police,-1.850037,52.492624,On or near Highfield Road,E01009484,Birmingham 048D,Violence and sexual offences,Action to be taken by another organisation,NaN
2,d56df0052bf702af8c297e7d2c370a08947a2e136ee54c...,2024-09,Warwickshire Police,Warwickshire Police,-1.804607,52.480898,On or near Masham Close,E01009510,Birmingham 056E,Other crime,Court result unavailable,NaN
3,fee7efa09879c034335eccc4e8881d1281cc79353f4b9c...,2024-09,Warwickshire Police,Warwickshire Police,-1.782446,52.460245,On or near Newark Croft,E01009314,Birmingham 069B,Public order,Status update unavailable,NaN
4,d5e8948096a728954ff7091657b6ce8edeed2fe77bf0f7...,2024-09,Warwickshire Police,Warwickshire Police,-1.881772,52.461033,On or near Woodfield Crescent,E01009367,Birmingham 083A,Violence and sexual offences,Status update unavailable,NaN


In [14]:
warwickshire_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 104145 entries, 0 to 104144
Data columns (total 12 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Crime ID               84260 non-null   str    
 1   Month                  104145 non-null  str    
 2   Reported by            104145 non-null  str    
 3   Falls within           104145 non-null  str    
 4   Longitude              103249 non-null  float64
 5   Latitude               103249 non-null  float64
 6   Location               104145 non-null  str    
 7   LSOA code              103249 non-null  str    
 8   LSOA name              103249 non-null  str    
 9   Crime type             104145 non-null  str    
 10  Last outcome category  84260 non-null   str    
 11  Context                0 non-null       float64
dtypes: float64(3), str(9)
memory usage: 9.5 MB


In [15]:
warwickshire_df.isna().sum()

Crime ID                  19885
Month                         0
Reported by                   0
Falls within                  0
Longitude                   896
Latitude                    896
Location                      0
LSOA code                   896
LSOA name                   896
Crime type                    0
Last outcome category     19885
Context                  104145
dtype: int64

In [16]:
warwickshire_df.describe()

,Longitude,Latitude,Context
count,103249.000000,103249.000000,0.0
mean,-1.516020,52.381722,NaN
std,0.161908,0.142447,NaN
min,-5.255307,50.215417,NaN
25%,-1.585931,52.281587,NaN
50%,-1.516080,52.372660,NaN
75%,-1.460924,52.515460,NaN
max,0.587416,55.003267,NaN


In [17]:
warwickshire_df_summary = warwickshire_df.nunique().reset_index(name='n_unique')
warwickshire_df_summary.columns = ['column', 'n_unique']

print(warwickshire_df_summary)

                   column  n_unique
0                Crime ID     84259
1                   Month        25
2             Reported by         1
3            Falls within         1
4               Longitude      8946
5                Latitude      8926
6                Location      6537
7               LSOA code      1051
8               LSOA name      1051
9              Crime type        14
10  Last outcome category        13
11                Context         0


In [18]:
for x in warwickshire_df['Crime type'].unique():
     print(x)

Public order
Violence and sexual offences
Other crime
Robbery
Vehicle crime
Other theft
Anti-social behaviour
Possession of weapons
Theft from the person
Criminal damage and arson
Burglary
Shoplifting
Drugs
Bicycle theft


In [19]:
for x in warwickshire_df['Last outcome category'].unique():
    print(x)

Status update unavailable
Action to be taken by another organisation
Court result unavailable
Unable to prosecute suspect
Investigation complete; no suspect identified
nan
Awaiting court outcome
Further investigation is not in the public interest
Local resolution
Offender given a caution
Further action is not in the public interest
Formal action is not in the public interest
Suspect charged as part of another case
Under investigation


### West Midlands

In [20]:
west_midlands_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 688744 entries, 0 to 688743
Data columns (total 12 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Crime ID               637214 non-null  str    
 1   Month                  688744 non-null  str    
 2   Reported by            688744 non-null  str    
 3   Falls within           688744 non-null  str    
 4   Longitude              688744 non-null  float64
 5   Latitude               688744 non-null  float64
 6   Location               688744 non-null  str    
 7   LSOA code              688744 non-null  str    
 8   LSOA name              688744 non-null  str    
 9   Crime type             688744 non-null  str    
 10  Last outcome category  637214 non-null  str    
 11  Context                0 non-null       float64
dtypes: float64(3), str(9)
memory usage: 63.1 MB


In [21]:
west_midlands_df.isna().sum()

Crime ID                  51530
Month                         0
Reported by                   0
Falls within                  0
Longitude                     0
Latitude                      0
Location                      0
LSOA code                     0
LSOA name                     0
Crime type                    0
Last outcome category     51530
Context                  688744
dtype: int64

In [22]:
west_midlands_df.describe()

,Longitude,Latitude,Context
count,688744.000000,688744.000000,0.0
mean,-1.897652,52.493844,NaN
std,0.176851,0.062464,NaN
min,-2.205059,52.348321,NaN
25%,-2.014281,52.446742,NaN
50%,-1.918583,52.485199,NaN
75%,-1.828099,52.536953,NaN
max,-1.431931,52.661081,NaN


In [23]:
west_midlands_df_summary = west_midlands_df.nunique().reset_index(name='n_unique')
west_midlands_df_summary.columns = ['column', 'n_unique']

print(west_midlands_df_summary)

                   column  n_unique
0                Crime ID    637214
1                   Month        25
2             Reported by         1
3            Falls within         1
4               Longitude     26270
5                Latitude     25454
6                Location     19316
7               LSOA code      1760
8               LSOA name      1760
9              Crime type        14
10  Last outcome category        13
11                Context         0


### West Yorkshire

In [24]:
west_yorkshire_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 632306 entries, 0 to 632305
Data columns (total 12 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Crime ID               574753 non-null  str    
 1   Month                  632306 non-null  str    
 2   Reported by            632306 non-null  str    
 3   Falls within           632306 non-null  str    
 4   Longitude              626393 non-null  float64
 5   Latitude               626393 non-null  float64
 6   Location               632306 non-null  str    
 7   LSOA code              626393 non-null  str    
 8   LSOA name              626393 non-null  str    
 9   Crime type             632306 non-null  str    
 10  Last outcome category  574753 non-null  str    
 11  Context                0 non-null       float64
dtypes: float64(3), str(9)
memory usage: 57.9 MB


In [25]:
west_yorkshire_df.isna().sum()

Crime ID                  57553
Month                         0
Reported by                   0
Falls within                  0
Longitude                  5913
Latitude                   5913
Location                      0
LSOA code                  5913
LSOA name                  5913
Crime type                    0
Last outcome category     57553
Context                  632306
dtype: int64

In [26]:
west_yorkshire_df.describe()

,Longitude,Latitude,Context
count,626393.000000,626393.000000,0.0
mean,-1.643278,53.758434,NaN
std,0.167002,0.070308,NaN
min,-2.734806,52.586004,NaN
25%,-1.774926,53.704821,NaN
50%,-1.636739,53.776510,NaN
75%,-1.527939,53.807300,NaN
max,-0.278988,54.821894,NaN


In [27]:
west_yorkshire_df_summary = west_yorkshire_df.nunique().reset_index(name='n_unique')
west_yorkshire_df_summary.columns = ['column', 'n_unique']

print(west_yorkshire_df_summary)

                   column  n_unique
0                Crime ID    574751
1                   Month        25
2             Reported by         1
3            Falls within         1
4               Longitude     33845
5                Latitude     32676
6                Location     24452
7               LSOA code      1450
8               LSOA name      1450
9              Crime type        14
10  Last outcome category        15
11                Context         0


In [28]:
for x in west_yorkshire_df['Crime type'].unique():
     print(x)

Anti-social behaviour
Criminal damage and arson
Violence and sexual offences
Other crime
Public order
Vehicle crime
Shoplifting
Robbery
Burglary
Other theft
Bicycle theft
Drugs
Theft from the person
Possession of weapons


In [29]:
for x in west_yorkshire_df['Last outcome category'].unique():
     print(x)

nan
Investigation complete; no suspect identified
Unable to prosecute suspect
Court result unavailable
Status update unavailable
Further action is not in the public interest
Local resolution
Further investigation is not in the public interest
Awaiting court outcome
Formal action is not in the public interest
Action to be taken by another organisation
Offender given a caution
Offender given penalty notice
Suspect charged as part of another case
Offender given a drugs possession warning
Under investigation


## Removing Null Locations & Null Column
Upon conclusion of the [Data Integrity Checking](##data-integrity-checking) section, it is noted that all police regions have missing values in the `Crime ID` and `Last outcome category` columns with the `Context` column being fully blank for all police regions. Additionaly, for all regions except for West Midlands, there exists some missing values in the `LSOA code`, `LSOA name`, `Longitude`, and `Latitude` columns. Also, since the missing values in the aforementioned four columns are equal for all regions, the project assumes that the columns are linked and a missing value in one of the fields will mean the other three are also missing values though this will be confirmed in the following section.

For the following section, this project elects to remove all rows with missing location data. This is because location data arguably the most important data for informing which locations are the most/least desirable and the dataset we are using would be useless for our purposes without it.

In terms of methodology for the following sectino, the same repitiion exists to ensure the aforementioned assumption that missing values in one of the location related fields equates to a missing value in the other three. The technique used here is that the script counts how many rows are remaining after dropping rows with missing values in `LSOA code`. This figure is then compared to the number of rows remaining after dropping the other three location data related columns with missing values. If these two are equal, then the assumption is correct.

`Context` column is also dropped as it is fully blank.

Missing values in `Crime ID` and `Last outcome category` are not removed as whether they are missing values or not is irrelvant for the purposes of this project.

In [30]:
def clean_null_locations(df):
    return df.dropna(subset=['LSOA code', 'LSOA name', 'Longitude', 'Latitude']).drop('Context', axis=1)

### Thames Valley

In [31]:
print(len(thames_valley_df))

print(len(thames_valley_df['LSOA code'].dropna()))

print(len(thames_valley_df.dropna(subset = ['LSOA name','Longitude','Latitude'])))

408443
399558
399558


In [32]:
no_null_LSOA_thames_valley_df = clean_null_locations(thames_valley_df)

In [33]:
no_null_LSOA_thames_valley_df.info()

<class 'pandas.DataFrame'>
Index: 399558 entries, 0 to 408029
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Crime ID               358420 non-null  str    
 1   Month                  399558 non-null  str    
 2   Reported by            399558 non-null  str    
 3   Falls within           399558 non-null  str    
 4   Longitude              399558 non-null  float64
 5   Latitude               399558 non-null  float64
 6   Location               399558 non-null  str    
 7   LSOA code              399558 non-null  str    
 8   LSOA name              399558 non-null  str    
 9   Crime type             399558 non-null  str    
 10  Last outcome category  358420 non-null  str    
dtypes: float64(2), str(9)
memory usage: 36.6 MB


### Warwickshire

In [34]:
print(len(warwickshire_df))

print(len(warwickshire_df['LSOA code'].dropna()))

print(len(warwickshire_df.dropna(subset = ['LSOA name','Longitude','Latitude'])))

104145
103249
103249


In [35]:
no_null_LSOA_warwickshire_df = clean_null_locations(warwickshire_df)

In [36]:
no_null_LSOA_warwickshire_df.info()

<class 'pandas.DataFrame'>
Index: 103249 entries, 0 to 104065
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Crime ID               83365 non-null   str    
 1   Month                  103249 non-null  str    
 2   Reported by            103249 non-null  str    
 3   Falls within           103249 non-null  str    
 4   Longitude              103249 non-null  float64
 5   Latitude               103249 non-null  float64
 6   Location               103249 non-null  str    
 7   LSOA code              103249 non-null  str    
 8   LSOA name              103249 non-null  str    
 9   Crime type             103249 non-null  str    
 10  Last outcome category  83365 non-null   str    
dtypes: float64(2), str(9)
memory usage: 9.5 MB


### West Midlands

In [37]:
print(len(west_midlands_df))

print(len(west_midlands_df['LSOA code'].dropna()))

print(len(west_midlands_df.dropna(subset = ['LSOA name','Longitude','Latitude'])))

688744
688744
688744


In [38]:
no_null_LSOA_west_midlands_df = clean_null_locations(west_midlands_df)

In [39]:
no_null_LSOA_west_midlands_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 688744 entries, 0 to 688743
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Crime ID               637214 non-null  str    
 1   Month                  688744 non-null  str    
 2   Reported by            688744 non-null  str    
 3   Falls within           688744 non-null  str    
 4   Longitude              688744 non-null  float64
 5   Latitude               688744 non-null  float64
 6   Location               688744 non-null  str    
 7   LSOA code              688744 non-null  str    
 8   LSOA name              688744 non-null  str    
 9   Crime type             688744 non-null  str    
 10  Last outcome category  637214 non-null  str    
dtypes: float64(2), str(9)
memory usage: 57.8 MB


### West Yorkshire

In [40]:
print(len(west_yorkshire_df))

print(len(west_yorkshire_df['LSOA code'].dropna()))

print(len(west_yorkshire_df.dropna(subset = ['LSOA name','Longitude','Latitude'])))

632306
626393
626393


In [41]:
no_null_LSOA_west_yorkshire_df = clean_null_locations(west_yorkshire_df)

In [42]:
no_null_LSOA_west_yorkshire_df.info()

<class 'pandas.DataFrame'>
Index: 626393 entries, 0 to 632016
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Crime ID               569163 non-null  str    
 1   Month                  626393 non-null  str    
 2   Reported by            626393 non-null  str    
 3   Falls within           626393 non-null  str    
 4   Longitude              626393 non-null  float64
 5   Latitude               626393 non-null  float64
 6   Location               626393 non-null  str    
 7   LSOA code              626393 non-null  str    
 8   LSOA name              626393 non-null  str    
 9   Crime type             626393 non-null  str    
 10  Last outcome category  569163 non-null  str    
dtypes: float64(2), str(9)
memory usage: 57.3 MB


## Outlier Checking
With our assumption confirmed and relevant missing values removed, outlier checks need to be done for the numerical columns of `Latitude` and `Longitude`. Due to the nature of this data, it is inaccurate to identify outliers using conventional statistical methods. In a situation where no readily available database exists to check against, eyeing the boundaries of a given police region and checking the minimum and maximum coordinate values is often the best available option. However, it was spotted that the range of coordinate values for crimes in the Warwickshire Police region were extremely large and spanned across the UK. As such, a more systematic approach needed to be used for filtering outliers.

The following section uses the publicly available [Police Force Areas](https://www.data.gov.uk/dataset/603056e4-58f7-4038-8259-70198410ea60/police-force-areas-december-2023-boundaries-ew-bfc) GeoJSON file to check whether coordinates recorded for a given crime for a given region actually lie within its boundaries. A sample of out of bound coordinates are also printed for manual checking if needed.

In [43]:
police_regions = gpd.read_file("Police_Force_Areas_December_2023_EW_BFC_-5569444588210716351.geojson")

police_regions = police_regions[police_regions["PFA23NM"].isin(["Thames Valley", "Warwickshire", "West Midlands", "West Yorkshire"])]

police_regions.head()

,FID,PFA23CD,PFA23NM,BNG_E,BNG_N,LONG,LAT,GlobalID,geometry
9,10,E23000010,West Yorkshire,418683,427238,-1.71822,53.74120,389db96e-b418-4f3b-81e3-4dcdc1e9ecbb,"POLYGON ((407885.502 451894.696, 407943.597 45..."
13,14,E23000014,West Midlands,402503,289718,-1.96454,52.50536,1ee318ae-5a70-4ad6-9c62-b952c414fbb1,"POLYGON ((403232.49 307228.27, 403253.79 30718..."
16,17,E23000017,Warwickshire,429584,253588,-1.56874,52.17977,a85b3aef-c835-4b33-ba08-1a15ead94682,"POLYGON ((428696.498 309045.196, 428700.596 30..."
28,29,E23000029,Thames Valley,462576,202253,-1.09561,51.71552,81ef12b5-c129-46f8-8d77-2b445aae58ec,"POLYGON ((490173.603 256107.999, 490234.897 25..."


"Police" is appended to each region for easy matching to our existing dataframes

In [44]:
police_regions['PFA23NM'] += " Police"

police_regions.head()

,FID,PFA23CD,PFA23NM,BNG_E,BNG_N,LONG,LAT,GlobalID,geometry
9,10,E23000010,West Yorkshire Police,418683,427238,-1.71822,53.74120,389db96e-b418-4f3b-81e3-4dcdc1e9ecbb,"POLYGON ((407885.502 451894.696, 407943.597 45..."
13,14,E23000014,West Midlands Police,402503,289718,-1.96454,52.50536,1ee318ae-5a70-4ad6-9c62-b952c414fbb1,"POLYGON ((403232.49 307228.27, 403253.79 30718..."
16,17,E23000017,Warwickshire Police,429584,253588,-1.56874,52.17977,a85b3aef-c835-4b33-ba08-1a15ead94682,"POLYGON ((428696.498 309045.196, 428700.596 30..."
28,29,E23000029,Thames Valley Police,462576,202253,-1.09561,51.71552,81ef12b5-c129-46f8-8d77-2b445aae58ec,"POLYGON ((490173.603 256107.999, 490234.897 25..."


In [45]:
# Function to check if points are within polygon
def check_points_in_region(df, region, lon_col="Longitude", lat_col="Latitude"):
    poly = police_regions.loc[police_regions["PFA23NM"] == region].geometry.iloc[0]
    # Create GeoDataFrame for points
    gdf_points = gpd.GeoDataFrame(df, geometry=[Point(xy) for xy in zip(df[lon_col], df[lat_col])], crs="EPSG:4326")
    # Transform to same CRS as polygons
    gdf_points = gdf_points.to_crs(police_regions.crs)
    # Check containment
    inside = gdf_points.geometry.within(poly)
    return inside.sum(), len(df), df[~inside].head()

# Check for each region
regions_check = {
    "Thames Valley Police": no_null_LSOA_thames_valley_df,
    "Warwickshire Police": no_null_LSOA_warwickshire_df,
    "West Midlands Police": no_null_LSOA_west_midlands_df,
    "West Yorkshire Police": no_null_LSOA_west_yorkshire_df
}

for region, df in regions_check.items():
    inside_count, total, outside_head = check_points_in_region(df, region)
    print(f"{region}: {inside_count}/{total} crimes are within the region, {total - inside_count} are outside the region")
    display(outside_head)

Thames Valley Police: 399543/399558 crimes are within the region, 15 are outside the region


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category
4567,4ff9c5818332b549137fc417cda44ff8424883bca912bc...,2024-09,Thames Valley Police,Thames Valley Police,-0.647295,51.813886,On or near Upper Icknield Way,E01023419,Dacorum 003B,Other theft,Investigation complete; no suspect identified
34205,6e77954fbb7ac2028e799ffbe16e54e25a728302889de2...,2024-06,Thames Valley Police,Thames Valley Police,-1.130019,51.358043,On or near Silchester Road,E01022555,Basingstoke and Deane 001C,Other theft,Unable to prosecute suspect
55670,3d231875fc1931f63a8b4dbf05adf132aeaf8e1bead110...,2024-01,Thames Valley Police,Thames Valley Police,-0.631931,52.080285,On or near Shire Lane,E01033814,Central Bedfordshire 007H,Other theft,Unable to prosecute suspect
71506,18136a6fa0f65bdc1a72f90eacb2fb87fcdd32b492fd84...,2024-08,Thames Valley Police,Thames Valley Police,-0.645094,52.010368,On or near Aspley Hill,E01017380,Central Bedfordshire 007B,Violence and sexual offences,Unable to prosecute suspect
72459,66282c5e5314d9d70ff32e62d96fd034afdd72ce08fe95...,2024-08,Thames Valley Police,Thames Valley Police,-0.647295,51.813886,On or near Upper Icknield Way,E01023419,Dacorum 003B,Other theft,Investigation complete; no suspect identified


Warwickshire Police: 101797/103249 crimes are within the region, 1452 are outside the region


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category
0,79d40667dd3ade5f202d0cfa5295777a869d9a30362fe2...,2024-09,Warwickshire Police,Warwickshire Police,-1.774147,52.494482,On or near Coneyford Road,E01009306,Birmingham 046B,Public order,Status update unavailable
1,236de2cd796df699d95982495b710ad2f35c4ac26db31f...,2024-09,Warwickshire Police,Warwickshire Police,-1.850037,52.492624,On or near Highfield Road,E01009484,Birmingham 048D,Violence and sexual offences,Action to be taken by another organisation
2,d56df0052bf702af8c297e7d2c370a08947a2e136ee54c...,2024-09,Warwickshire Police,Warwickshire Police,-1.804607,52.480898,On or near Masham Close,E01009510,Birmingham 056E,Other crime,Court result unavailable
3,fee7efa09879c034335eccc4e8881d1281cc79353f4b9c...,2024-09,Warwickshire Police,Warwickshire Police,-1.782446,52.460245,On or near Newark Croft,E01009314,Birmingham 069B,Public order,Status update unavailable
4,d5e8948096a728954ff7091657b6ce8edeed2fe77bf0f7...,2024-09,Warwickshire Police,Warwickshire Police,-1.881772,52.461033,On or near Woodfield Crescent,E01009367,Birmingham 083A,Violence and sexual offences,Status update unavailable


West Midlands Police: 688192/688744 crimes are within the region, 552 are outside the region


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category
106,NaN,2024-09,West Midlands Police,West Midlands Police,-1.857073,52.598426,On or near Hornton Close,E01009421,Birmingham 004A,Anti-social behaviour,NaN
12755,f789a9e8baa28d656b5f1bb7c82f3057eb855c16d88db0...,2024-09,West Midlands Police,West Midlands Police,-1.851428,52.394086,On or near Rosebriars,E01032146,Bromsgrove 005B,Vehicle crime,Court result unavailable
18414,NaN,2024-09,West Midlands Police,West Midlands Police,-1.922464,52.656048,On or near A5195,E01034104,Lichfield 006G,Anti-social behaviour,NaN
18415,26d6664f638e6d2a30020e12960ba3fe63d6cd7f2c1c8c...,2024-09,West Midlands Police,West Midlands Police,-1.922464,52.656048,On or near A5195,E01034104,Lichfield 006G,Violence and sexual offences,Unable to prosecute suspect
18416,db8be3be4326291a1c6378fd36d31b2560ca52bb3a1f6d...,2024-09,West Midlands Police,West Midlands Police,-1.908322,52.653134,On or near Barracks Lane,E01029505,Lichfield 009E,Criminal damage and arson,Local resolution


West Yorkshire Police: 626156/626393 crimes are within the region, 237 are outside the region


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category
0,NaN,2024-09,West Yorkshire Police,West Yorkshire Police,-1.472961,53.599861,On or near Railway Walk,E01007439,Barnsley 001D,Anti-social behaviour,NaN
1,f25c85f13cecad005f8ad648e049e9eb3f55db06e463e0...,2024-09,West Yorkshire Police,West Yorkshire Police,-1.732999,53.545103,On or near Wood Royd Hill Lane,E01007426,Barnsley 027D,Criminal damage and arson,Investigation complete; no suspect identified
8432,NaN,2024-09,West Yorkshire Police,West Yorkshire Police,-1.597499,54.821894,On or near Hawthorn Close,E01020607,County Durham 019B,Anti-social behaviour,NaN
8433,NaN,2024-09,West Yorkshire Police,West Yorkshire Police,-1.597499,54.821894,On or near Hawthorn Close,E01020607,County Durham 019B,Anti-social behaviour,NaN
21372,NaN,2024-09,West Yorkshire Police,West Yorkshire Police,-1.318479,53.899980,On or near Fairfax Gardens,E01027922,Selby 001E,Anti-social behaviour,NaN


## Removing Out of Bound Locations
Upon highlighting all out of bounds coordinate values, it is noted that though these entries exist in significant quantity to justify the need for removal, they comprise an insignificant proportion of our overall data.

The script in the following section will remove these entries anyways to prevent issues that may occur when plotting crime data using a map format.

In [46]:
# Function to remove points outside the region
def remove_points_outside_region(df, region, lon_col="Longitude", lat_col="Latitude"):
    poly = police_regions.loc[police_regions["PFA23NM"] == region].geometry.iloc[0]
    # Create GeoDataFrame for points
    gdf_points = gpd.GeoDataFrame(df, geometry=[Point(xy) for xy in zip(df[lon_col], df[lat_col])], crs="EPSG:4326")
    # Transform to same CRS as polygons
    gdf_points = gdf_points.to_crs(police_regions.crs)
    # Filter to points inside
    inside_mask = gdf_points.geometry.within(poly)
    # Return the filtered dataframe without geometry column
    return df[inside_mask].reset_index()

# Apply to each region
clean_thames_valley_df = remove_points_outside_region(no_null_LSOA_thames_valley_df, "Thames Valley Police")
clean_warwickshire_df = remove_points_outside_region(no_null_LSOA_warwickshire_df, "Warwickshire Police")
clean_west_midlands_df = remove_points_outside_region(no_null_LSOA_west_midlands_df, "West Midlands Police")
clean_west_yorkshire_df = remove_points_outside_region(no_null_LSOA_west_yorkshire_df, "West Yorkshire Police")

print("Filtered dataframes created:")
print(f"Thames Valley: {len(clean_thames_valley_df)} rows")
print(f"Warwickshire: {len(clean_warwickshire_df)} rows")
print(f"West Midlands: {len(clean_west_midlands_df)} rows")
print(f"West Yorkshire: {len(clean_west_yorkshire_df)} rows")

Filtered dataframes created:
Thames Valley: 399543 rows
Warwickshire: 101797 rows
West Midlands: 688192 rows
West Yorkshire: 626156 rows


## Aggregation and Checking of Clean Data
With the data successfully cleaned, only now will the data be aggregated for exporting. Some final checks are done here to ensure there are no issues during aggregation.

In [47]:
clean_dataset_week6_df = pd.concat([clean_thames_valley_df,
                                    clean_warwickshire_df,
                                    clean_west_midlands_df,
                                    clean_west_yorkshire_df
                                    ], ignore_index=True)

In [48]:
clean_dataset_week6_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1815688 entries, 0 to 1815687
Data columns (total 12 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   index                  int64  
 1   Crime ID               str    
 2   Month                  str    
 3   Reported by            str    
 4   Falls within           str    
 5   Longitude              float64
 6   Latitude               float64
 7   Location               str    
 8   LSOA code              str    
 9   LSOA name              str    
 10  Crime type             str    
 11  Last outcome category  str    
dtypes: float64(2), int64(1), str(9)
memory usage: 166.2 MB


In [49]:
clean_dataset_week6_df.head()

,index,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category
0,0,600dc88bebb3499405cf71237c6a89fa4f0e94f51d669d...,2024-09,Thames Valley Police,Thames Valley Police,-0.704117,51.448857,On or near Winkfield Lane,E01016252,Bracknell Forest 001C,Burglary,Investigation complete; no suspect identified
1,1,96339937ce33810cdd4421d7960675e206d9a57e8cddd7...,2024-09,Thames Valley Police,Thames Valley Police,-0.671521,51.446882,On or near Montague Park,E01016252,Bracknell Forest 001C,Other theft,Unable to prosecute suspect
2,2,48064c2c5c76557638c7d6e9351d59dd3c6a9105a042d5...,2024-09,Thames Valley Police,Thames Valley Police,-0.667887,51.450114,On or near Park Lane,E01016252,Bracknell Forest 001C,Other theft,Status update unavailable
3,3,b88c4f1501e108884efed2d65a1affd624539250adb7c9...,2024-09,Thames Valley Police,Thames Valley Police,-0.672384,51.435490,On or near Lovel Lane,E01016252,Bracknell Forest 001C,Other theft,Investigation complete; no suspect identified
4,4,a8481f9aa2db34e456f19bdbd0620a9e646cb7f98d55a9...,2024-09,Thames Valley Police,Thames Valley Police,-0.677236,51.436336,On or near Lovel Road,E01016252,Bracknell Forest 001C,Violence and sexual offences,Unable to prosecute suspect


In [50]:
clean_dataset_week6_df.tail()

,index,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category
1815683,632012,9da3e221b0afcfb0136675053e44eafd378f8002f4d970...,2024-10,West Yorkshire Police,West Yorkshire Police,-1.329551,53.592690,On or near Hemsworth Road,E01011872,Wakefield 045D,Violence and sexual offences,Unable to prosecute suspect
1815684,632013,1afe1977640a6fc1bb89b15486c2187d33b6f9e3647f8c...,2024-10,West Yorkshire Police,West Yorkshire Police,-1.326375,53.590048,On or near Grove Way,E01011872,Wakefield 045D,Violence and sexual offences,Court result unavailable
1815685,632014,90dae2771a4437b1d9bc48ccd43bba39e86c38adac027f...,2024-10,West Yorkshire Police,West Yorkshire Police,-1.329551,53.592690,On or near Hemsworth Road,E01011872,Wakefield 045D,Violence and sexual offences,Unable to prosecute suspect
1815686,632015,8ccc290b3086a68d8d9df67b1714b84fadcfe2cc19f2eb...,2024-10,West Yorkshire Police,West Yorkshire Police,-1.331430,53.590426,On or near Marion Close,E01011872,Wakefield 045D,Violence and sexual offences,Unable to prosecute suspect
1815687,632016,b2e9796bc6843eb148510afee0ec2c444c4e94dd1e1856...,2024-10,West Yorkshire Police,West Yorkshire Police,-1.327834,53.587584,On or near Radford Park Avenue,E01011872,Wakefield 045D,Other crime,Unable to prosecute suspect


In [51]:
clean_dataset_week6_df.isna().sum()

index                         0
Crime ID                 169530
Month                         0
Reported by                   0
Falls within                  0
Longitude                     0
Latitude                      0
Location                      0
LSOA code                     0
LSOA name                     0
Crime type                    0
Last outcome category    169530
dtype: int64

In [52]:
clean_dataset_week6_df.describe()

,index,Longitude,Latitude
count,1.815688e+06,1.815688e+06,1.815688e+06
mean,2.873397e+05,-1.575918e+00,5.274426e+01
std,1.890396e+05,4.138830e-01,8.067987e-01
min,0.000000e+00,-2.201955e+00,5.133634e+01
25%,1.186240e+05,-1.890885e+00,5.237457e+01
50%,2.714510e+05,-1.671550e+00,5.250811e+01
75%,4.326112e+05,-1.401899e+00,5.371158e+01
max,6.887430e+05,-4.814580e-01,5.395762e+01


In [53]:
clean_dataset_week6_df.nunique()

index                     688702
Crime ID                 1646155
Month                         25
Reported by                    4
Falls within                   4
Longitude                  96107
Latitude                   95127
Location                   60034
LSOA code                   4965
LSOA name                   4965
Crime type                    14
Last outcome category         15
dtype: int64

In [54]:
for x in clean_dataset_week6_df['Month'].sort_values().unique():
     print(x)

2024-01
2024-02
2024-03
2024-04
2024-05
2024-06
2024-07
2024-08
2024-09
2024-10
2024-11
2024-12
2025-01
2025-02
2025-03
2025-04
2025-05
2025-06
2025-07
2025-08
2025-09
2025-10
2025-11
2025-12
2026-01


In [55]:
for x in clean_dataset_week6_df['Crime type'].unique():
     print(x)

Burglary
Other theft
Violence and sexual offences
Possession of weapons
Other crime
Public order
Criminal damage and arson
Anti-social behaviour
Bicycle theft
Drugs
Shoplifting
Robbery
Vehicle crime
Theft from the person


In [56]:
for x in clean_dataset_week6_df['Last outcome category'].unique():
     print(x)

Investigation complete; no suspect identified
Unable to prosecute suspect
Status update unavailable
Action to be taken by another organisation
Awaiting court outcome
Court result unavailable
nan
Further investigation is not in the public interest
Local resolution
Offender given a caution
Suspect charged as part of another case
Further action is not in the public interest
Offender given penalty notice
Formal action is not in the public interest
Under investigation
Offender given a drugs possession warning


## Clean Data Exporting

In [57]:
# clean_dataset_week6_df.to_csv("clean_dataset_week6.csv")